In [1]:
# Here we try to implement SSL pretraining 
# with the downstream evaluation callback
# that will fully retrain the probing model
# before evaluating it after each SSL epoch.

# Also, SSL pretraining is at the level of individual MS1 spectra.
# Evaluation is at the run level and based on run-level embeddings
# (run embedding = average of all MS1 embeddings)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!nvidia-smi

Wed Mar 18 21:28:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:4A:00.0 Off |                    0 |
| N/A   28C    P8             33W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Put project root on sys.path so "source" is importable
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent  # notebooks/ -> root/
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

/home/mpominova/lcms-foundation-model


In [4]:
import os
import yaml
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
import pytorch_lightning as L
from tqdm import tqdm
from depthcharge.data import SpectrumDataset, spectra_to_df, preprocessing
# from depthcharge.tokenizers import Tokenizer
from torch.utils.data import DataLoader

from source.dataset import LanceMapDataset, RunDataset
from source.model import MS1Encoder
from source.scheduler import CosineWarmupScheduler
from source.config import ExperimentConfig, DataConfig, ModelConfig, OptimizerConfig, TrainingConfig

/home/mpominova/.local/share/mamba/envs/lcms-foundation/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# last version if MS1Encoder model source code # 12/03/26 

import numpy as np
import pandas as pd # DEBUG
import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as L
from depthcharge.encoders import PeakEncoder, PositionalEncoder
from depthcharge.transformers import SpectrumTransformerEncoder

from IPython.display import clear_output # DEBUG
pd.set_option('display.max_rows', 500) # DEBUG

class MS1Encoder(L.LightningModule):
    def __init__(
        self,
        d_model=128,
        nhead=8,
        dim_feedforward=512,
        n_layers=4,
        dropout=0.1,
        n_bins=2000,
        bin_mz_min=0,
        bin_mz_max=2000,
        masked_peaks_fraction=0.3,
        mask_proportional=True,
        lr=5e-4,
        warmup_iters=1000,
        cosine_schedule_period_iters=32000,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.d_model = d_model
        self.nhead = nhead
        self.dim_feedforward = dim_feedforward
        self.n_layers = n_layers
        self.dropout = dropout

        self.n_bins = n_bins
        self.bin_mz_min = bin_mz_min
        self.bin_mz_max = bin_mz_max
        self.masked_peaks_fraction = masked_peaks_fraction
        self.mask_proportional = mask_proportional

        self.peak_encoder = nn.Sequential(
            PeakEncoder(
                d_model=self.d_model,
                min_mz_wavelength=0.001,
                max_mz_wavelength=10000,
                min_intensity_wavelength=1e-06,
                max_intensity_wavelength=1,
                learnable_wavelengths=False,
            ),
        )
        self.encoder = SpectrumTransformerEncoder(
            d_model=self.d_model,
            nhead=self.nhead,
            dim_feedforward=self.dim_feedforward,  # 1024,
            n_layers=self.n_layers,
            dropout=self.dropout,
            peak_encoder=self.peak_encoder,
        )

        self.head_mz = nn.Sequential(
            nn.Linear(d_model, n_bins),
        )  # outputs n_bins logits for each peak
        self.head_I = nn.Sequential(
            nn.Linear(d_model, 1),
        )  # outputs float I value for each peak

        # losses
        self.loss_mz_bin = nn.CrossEntropyLoss(reduction="mean", ignore_index=-1)
        self.loss_I = nn.MSELoss(reduction="mean")
        # metrics
        self.train_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)
        self.val_accuracy_mz_bin = torchmetrics.classification.Accuracy(task="multiclass", num_classes=self.n_bins, ignore_index=-1)

    def get_peaks_mask(self, intensities, proportional=False):
        if proportional:
            k = int(intensities.size(1) * self.masked_peaks_fraction)
            mask = torch.zeros_like(intensities, dtype=torch.bool)
            # FIXME: assume we never have zero rows (= spectra with no peaks), otherwise will fail

            # compute sampling weights w
            I_mean = intensities.sum(dim=1) / (intensities != 0).sum(dim=1) # mean I of non-zero peaks
            w = intensities + (intensities == 0).float() * I_mean[:, None]  # weight 0s by mean I
            # sample k indices without replacement, weighted by w
            idx = torch.multinomial(w, num_samples=k, replacement=False)
            mask = mask.scatter(dim=1, index=idx.to(dtype=torch.int64), value=True) # value to write into mask (True)

        else:
            mask = torch.rand_like(intensities) < self.masked_peaks_fraction
        return mask

    def get_mz_bins(self, mz):
        # every peak with mz > bin_mz_max will belong to max bin
        mz = mz.clamp(0, self.bin_mz_max - 1)
        mz_binned = (
            ((mz - self.bin_mz_min) / (self.bin_mz_max - self.bin_mz_min) * self.n_bins)
            .floor()
            .long()
        )
        mz_binned[mz < self.bin_mz_min] = -1
        return mz_binned

    def forward(
        self,
        mzs: torch.Tensor,
        intensities: torch.Tensor,
    ):
        peak_embs, _ = self.encoder(mz_array=mzs, intensity_array=intensities)
        # drop global token
        peak_embs = peak_embs[:, 1:, :]
        return peak_embs

    # def ssl_step(self):
    #     # TODO: move here the repeated part of training & validation parts
    #     return

    def training_step(self, batch, batch_idx):
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("train_loss_mz_bin", loss_mz_bin.item())
        # self.log("train_loss_I", loss_I.item())
        self.log("train_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.train_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("train_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=True, on_epoch=False)
        return loss

    def validation_step(self, batch, batch_idx):
        # Note: validation is now always partially random (mask sampling)
        # so not identical between epochs. May want to change it (how?).
        mz = batch["mz_array"]
        I = batch["intensity_array"]

        # sample peak masks
        masks = self.get_peaks_mask(I, proportional=self.mask_proportional)

        # prepare targets (bins & I of masked peaks)
        target_mz, target_I = mz[masks], I[masks]
        # transform mz into bins (target classes C \in [0, n_bins - 1])
        target_mz_bins = self.get_mz_bins(target_mz)

        # mask input peaks with 0 (before encoding)
        masked_mz = mz * (1 - masks.float())
        # masked_I = I * (1 - masks.float()) # FIX: not mask intensities, only mz
        
        # get embeddings for all peaks
        # peak_embs = self.forward(masked_mz, masked_I)
        peak_embs = self.forward(masked_mz, I) # FIX: not mask intensities, only mz
        # select only embeddings of masked peaks
        masked_peak_embs = peak_embs[masks]
        # predict masked peaks binned mz & I
        pred_mz_bins = self.head_mz(masked_peak_embs)
        pred_I = self.head_I(masked_peak_embs).squeeze(dim=-1)

        loss_mz_bin = self.loss_mz_bin(pred_mz_bins, target_mz_bins)
        # loss_I = self.loss_I(pred_I, target_I)
        loss = loss_mz_bin# + loss_I
        self.log("val_loss_mz_bin", loss_mz_bin.item())
        # self.log("val_loss_I", loss_I.item())
        self.log("val_loss", loss.item())
        # Accuracy metric for mz bin prediction
        acc_mz_bin = self.val_accuracy_mz_bin(pred_mz_bins, target_mz_bins)
        self.log("val_acc_mz_bin", acc_mz_bin.item(), prog_bar=True, on_step=False, on_epoch=True)

        # # Display prediction sample
        # n = 30
        # mz_bins_true, I_true = target_mz_bins[:n].cpu().numpy(), target_I[:n].cpu().numpy()
        # mz_bins_pred, I_pred = pred_mz_bins[:n].argmax(dim=1).cpu().numpy(), pred_I[:n].cpu().numpy()
        # sample_df = np.column_stack((
        #     mz_bins_true.ravel(), 
        #     I_true.ravel(), 
        #     mz_bins_pred.ravel(), 
        #     # I_pred.ravel()
        # ))
        # sample_df = pd.DataFrame(
        #     sample_df, columns=[
        #         "mz_bins_true", 
        #         "I_true", 
        #         "mz_bins_pred", 
        #         # "I_pred"
        #     ]
        # )
        # display(sample_df)
        
        return loss

    def on_validation_epoch_start(self,):
        clear_output(True)
        return

    def configure_optimizers(
        self,
    ):
        """TODO."""
        optimizer = torch.optim.Adam(
            self.parameters(), lr=self.hparams.lr, betas=(0.9, 0.98)
        )
        self.lr_scheduler = CosineWarmupScheduler(
            optimizer,
            self.hparams.warmup_iters,
            self.hparams.cosine_schedule_period_iters,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": self.lr_scheduler,
                "interval": "step",
                "frequency": 1,
                "name": "cosine_warmup",
            },
        }

    def optimizer_step(self, *args, **kwargs):
        super().optimizer_step(*args, **kwargs)
        self.log("lr", self.lr_scheduler.get_last_lr()[0])

In [6]:
def load_config(config_path):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)
        config = ExperimentConfig(
            name=config_dict['name'],
            data=DataConfig(**config_dict['data']),
            model=ModelConfig(**config_dict['model']),
            optimizer=OptimizerConfig(**config_dict['optimizer']),
            training=TrainingConfig(**config_dict['training'])
        )
    return config

In [7]:
# Load training data
data_root_dir = "/mnt/data/shared/lc_ms_foundation/"
dset_name = "abele_data"
data_dir = os.path.join(data_root_dir, dset_name, "mzml")

mzml_files = os.listdir(data_dir)
print("Total N files:", len(mzml_files))

Total N files: 1334


In [8]:
meta_df = pl.read_csv("../data/filtered_abele_metadata.csv")
# meta_df = pl.read_csv(os.path.join(
#     data_root_dir, 
#     dset_name,
#     "all_abele_metadata.csv"
# ))

meta_df = meta_df.rename({
    "characteristics[organism]": "organism",
    "comment[data file]": "data_file"
})
meta_df

organism,genus,data_file
str,str,str
"""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026"""
"""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027"""
"""Micrococcus cohnii""","""Micrococcus""","""BBM_441_P110_31_MIA_032"""
"""Micrococcus lylae""","""Micrococcus""","""BBM_446_P110_31_MIA_052"""
"""Listeria seeligeri""","""Listeria""","""BBM_446_P110_31_MIA_057"""
…,…,…
"""Buttiauxella noackiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_010_10"""
"""Buttiauxella warmboldiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_011_10"""
"""Buttiauxella ferragutiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_013_10"""


In [9]:
# надо имплементировать retrain-evaluation
# и поставить ее тестовый прогон на СТАРЫХ данных

# так что ура, можем пока не думать и использовать то, что и было

### Preload data|

In [9]:
# we only want to load our previous subset of data (60 mzmls),
# not the entire dataset of 1334 files.
# so need to find exactly those 60 files. 

# we can simply take the files that are in old meta file.

genus_class = {
    "Listeria": 0,
    "Micrococcus": 1,
    "Shigella": 2,
    "Buttiauxella": 3,
}
meta_df = meta_df.with_columns((pl.col("data_file") + ".mzML").alias("peak_file"))
meta_df = meta_df.with_columns(
    (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))
)

display(meta_df["genus_class"].value_counts())
meta_df

/tmp/ipykernel_54842/1256098967.py:15: DeprecationWarning: the `return_dtype` parameter for `replace` is deprecated. Use `replace_strict` instead to set a return data type while replacing values.
(Deprecated in version 1.0.0)
  (pl.col("genus").replace(genus_class, return_dtype=int).alias("genus_class"))


genus_class,count
i64,u32
3,21
2,9
1,15
0,15


organism,genus,data_file,peak_file,genus_class
str,str,str,str,i64
"""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""","""BBM_437_P110_31_MIA_026.mzML""",2
"""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""","""BBM_441_P110_31_MIA_027.mzML""",0
"""Micrococcus cohnii""","""Micrococcus""","""BBM_441_P110_31_MIA_032""","""BBM_441_P110_31_MIA_032.mzML""",1
"""Micrococcus lylae""","""Micrococcus""","""BBM_446_P110_31_MIA_052""","""BBM_446_P110_31_MIA_052.mzML""",1
"""Listeria seeligeri""","""Listeria""","""BBM_446_P110_31_MIA_057""","""BBM_446_P110_31_MIA_057.mzML""",0
…,…,…,…,…
"""Buttiauxella noackiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_010_10""","""BBM_436_P110_31_MIA_010_10.mzM…",3
"""Buttiauxella warmboldiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_011_10""","""BBM_436_P110_31_MIA_011_10.mzM…",3
"""Buttiauxella ferragutiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_013_10""","""BBM_436_P110_31_MIA_013_10.mzM…",3


In [10]:
dset_mzml_files = meta_df["peak_file"].to_list()
len(dset_mzml_files)

60

In [11]:
# Use our "default" parameters from config to load data

# TODO: do we maybe need separate configs for different types of experiments?
# (e.g. MS2, MS1, online eval, retrain eval, run, ... ?)
# TODO: would be nice to fix some convenient naming convention for experiments

config = load_config("../config.yaml")
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [12]:
preprocessing_fn = [
    preprocessing.filter_intensity(max_num_peaks=config.data.max_num_peaks),
    preprocessing.scale_intensity(scaling="root", max_intensity=1.),
]

In [13]:
# TODO: spectra_to_df не извлекает RT по дефолту, 
# - но может быть можно это переопределить через custom fields

dfs = {
    mzml_file: spectra_to_df(
        os.path.join(data_dir, mzml_file),
        metadata_df=None,
        ms_level=1,
        preprocessing_fn=preprocessing_fn,
        valid_charge=None,
        custom_fields=None,
        progress=True,
    )
    for mzml_file in dset_mzml_files
}
len(dfs)

BBM_436_P110_31_MIA_022_10.mzML: 100%|██████████| 16168/16168 [00:06<00:00, 2317.51 spectra/s]


60

In [14]:
# split into train/probe_train/probe_val

In [15]:
def get_gradient_lenght(run_name):
    run_name = run_name.split("_")
    if len(run_name) < 7:
        return ""
    return run_name[6]

In [16]:
meta_df = meta_df.with_columns(
    (pl.col("data_file").str.split("_").list[1]).alias("id1")
)
meta_df = meta_df.with_columns(
    (pl.col("data_file").str.split("_").list[5]).alias("id2")
)
display(meta_df["id2"].value_counts())

meta_df = meta_df.with_columns(
    (pl.col("data_file").map_elements(get_gradient_lenght)).alias("grad_len")
)
display(meta_df["grad_len"].value_counts())

id2,count
str,u32
"""027""",1
"""059""",3
"""072""",2
"""057""",3
"""008""",3
…,…
"""017""",3
"""013""",3
"""002""",3


/tmp/ipykernel_54842/1693487057.py:9: MapWithoutReturnDtypeWarning: 'return_dtype' of function python_udf must be set

A later expression might fail because the output type is not known. Set return_dtype=pl.self_dtype() if the type is unchanged, or set the proper output data type.
  meta_df = meta_df.with_columns(


grad_len,count
str,u32
"""30""",20
"""""",20
"""10""",20


In [20]:
# [Split 4] 
# take classes 2+3 (9+21 samples) as SSL train
# uniform split classes 0+1 (15+15 samples) into probe_train/probe_val

In [17]:
meta_df = meta_df.with_columns(
    (pl.col("id1") + "_" + pl.col("id2")).alias("id")
)
meta_df["id"].value_counts()

id,count
str,u32
"""436_002""",2
"""436_009""",2
"""445_060""",2
"""445_052""",2
"""437_017""",1
…,…
"""437_022""",1
"""437_004""",1
"""437_011""",1


In [19]:
g = meta_df.select(["id", "genus_class"]).group_by("id")
sorted_ids = g.first().sort(by=["genus_class", "id"])["genus_class", "id"]

train_ids = sorted_ids.filter(sorted_ids["genus_class"].is_in([2, 3]))["id"].to_list()
val_ids = [i for i in sorted_ids["id"] if i not in train_ids]

probe_train_ids = val_ids[::2]
probe_val_ids = [i for i in val_ids if i not in probe_train_ids]

len(train_ids), len(probe_train_ids), len(probe_val_ids)

(20, 10, 10)

In [20]:
display(sorted_ids.filter(sorted_ids["id"].is_in(probe_train_ids)))
display(sorted_ids.filter(sorted_ids["id"].is_in(probe_val_ids)))

genus_class,id
i64,str
0,"""441_027"""
0,"""445_059"""
0,"""445_072"""
0,"""446_059"""
0,"""458_037"""
1,"""440_032"""
1,"""445_052"""
1,"""458_026"""
1,"""458_083"""


genus_class,id
i64,str
0,"""445_057"""
0,"""445_060"""
0,"""446_057"""
0,"""446_060"""
0,"""459_037"""
1,"""441_032"""
1,"""446_052"""
1,"""458_062"""
1,"""459_026"""


In [21]:
id2split = {}
for i in train_ids:
    id2split[i] = "train"
for i in probe_train_ids:
    id2split[i] = "probe_train"
for i in probe_val_ids:
    id2split[i] = "probe_val"
meta_df = meta_df.with_columns(
    pl.col("id").replace(id2split).alias("split")
)
meta_df

organism,genus,data_file,peak_file,genus_class,id1,id2,grad_len,id,split
str,str,str,str,i64,str,str,str,str,str
"""Shigella sonnei""","""Shigella""","""BBM_437_P110_31_MIA_026""","""BBM_437_P110_31_MIA_026.mzML""",2,"""437""","""026""","""""","""437_026""","""train"""
"""Listeria grayi""","""Listeria""","""BBM_441_P110_31_MIA_027""","""BBM_441_P110_31_MIA_027.mzML""",0,"""441""","""027""","""""","""441_027""","""probe_train"""
"""Micrococcus cohnii""","""Micrococcus""","""BBM_441_P110_31_MIA_032""","""BBM_441_P110_31_MIA_032.mzML""",1,"""441""","""032""","""""","""441_032""","""probe_val"""
"""Micrococcus lylae""","""Micrococcus""","""BBM_446_P110_31_MIA_052""","""BBM_446_P110_31_MIA_052.mzML""",1,"""446""","""052""","""""","""446_052""","""probe_val"""
"""Listeria seeligeri""","""Listeria""","""BBM_446_P110_31_MIA_057""","""BBM_446_P110_31_MIA_057.mzML""",0,"""446""","""057""","""""","""446_057""","""probe_val"""
…,…,…,…,…,…,…,…,…,…
"""Buttiauxella noackiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_010_10""","""BBM_436_P110_31_MIA_010_10.mzM…",3,"""436""","""010""","""10""","""436_010""","""train"""
"""Buttiauxella warmboldiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_011_10""","""BBM_436_P110_31_MIA_011_10.mzM…",3,"""436""","""011""","""10""","""436_011""","""train"""
"""Buttiauxella ferragutiae""","""Buttiauxella""","""BBM_436_P110_31_MIA_013_10""","""BBM_436_P110_31_MIA_013_10.mzM…",3,"""436""","""013""","""10""","""436_013""","""train"""


In [22]:
# Data for SSL training
split_df = meta_df.filter(pl.col("split") == "train")
print("SSL train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model training
split_df = meta_df.filter(pl.col("split") == "probe_train")
print("Downstream train - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

# Data for downstream model validation
split_df = meta_df.filter(pl.col("split") == "probe_val")
print("Downstream val - Total:", len(split_df))
display(split_df["genus_class"].value_counts(normalize=False).sort(by="genus_class"))
# display(split_df["genus_class"].value_counts(normalize=True).sort(by="genus_class"))

SSL train - Total: 30


genus_class,count
i64,u32
2,9
3,21


Downstream train - Total: 17


genus_class,count
i64,u32
0,8
1,9


Downstream val - Total: 13


genus_class,count
i64,u32
0,7
1,6


In [23]:
# train SpectrumDataset
train_df = pl.concat(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "train")["peak_file"].to_list()
    ], 
    how="vertical"
)
train_df = train_df.join(meta_df, on="peak_file", how="left")

# val SpectrumDataset
val_df = pl.concat(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(
            pl.col("split").is_in(["probe_train", "probe_val"])
        )["peak_file"].to_list()
    ], 
    how="vertical"
)
val_df = val_df.join(meta_df, on="peak_file", how="left")

train_df.shape, val_df.shape

((87385, 16), (87112, 16))

In [24]:
train_stream = SpectrumDataset(
    train_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", train_stream.n_spectra)
train_lance_path = str(train_stream.path)   # this is the key
print("Lance dataset path:", train_lance_path)
train_dataset = LanceMapDataset(train_lance_path, seq_len=config.data.max_num_peaks)

val_stream = SpectrumDataset(
    val_df.select(["mz_array", "intensity_array", "genus_class"]), 
    batch_size=256, # streaming batch size, not training batch size
)
print("N train spectra", val_stream.n_spectra)
val_lance_path = str(val_stream.path)   # this is the key
print("Lance dataset path:", val_lance_path)
val_dataset = LanceMapDataset(val_lance_path, seq_len=config.data.max_num_peaks)

N train spectra 87385
Lance dataset path: /tmp/tmpbidhigoo/361c11f2-b815-4aec-8a1d-f55fcc2b193b.lance
N train spectra 87112
Lance dataset path: /tmp/tmpkiwzidyp/d82d2268-a8a4-444a-a3b7-48021929ee3c.lance


In [25]:
run_labels = dict(zip(meta_df["peak_file"], meta_df["genus_class"]))

probe_train_dataset = RunDataset(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_train")["peak_file"].to_list()
    ], 
    run_labels=run_labels, 
    seq_len=config.data.max_num_peaks
)
probe_val_dataset = RunDataset(
    [
        dfs[mzml_file] for mzml_file 
        in meta_df.filter(pl.col("split") == "probe_val")["peak_file"].to_list()
    ], 
    run_labels=run_labels,
    seq_len=config.data.max_num_peaks
)
len(probe_train_dataset), len(probe_val_dataset)

100%|██████████| 13/13 [00:00<00:00, 55.91it/s]


(17, 13)

In [26]:
def run_collate_fn(rows):
    # rows is a list of dicts, one per spectrum
    keys = rows[0].keys()
    batch = {}
    for key in keys:
        if key in ['mz_array', 'intensity_array']:
            batch[key] = [torch.tensor(r[key]) for r in rows]
        else:
            batch[key] = torch.tensor([r[key] for r in rows])
    return batch

In [27]:
BATCH_SIZE = config.data.batch_size

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False
)
print("SSL:", len(train_loader), len(val_loader))

probe_train_loader = DataLoader(
    probe_train_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=True,
    collate_fn=run_collate_fn,
)
probe_val_loader = DataLoader(
    probe_val_dataset, 
    batch_size=BATCH_SIZE, 
    num_workers=0, 
    shuffle=False,
    collate_fn=run_collate_fn,
)
print("Downstream:", len(probe_train_loader), len(probe_val_loader))

SSL: 44 44
Downstream: 1 1


In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torchmetrics.functional import accuracy


# Full-retrain fine-tuner model
# Is applied at the end of every SSL validation epoch (Or every n epochs?)
# Every time before evaluation the model (needs to be with a fixed random seed?)
# is re-initialized and trained from scratch for probe_n_epochs (configured in init)


class FineTuner(L.Callback):
    def __init__(
        self,
        encoder_output_dim: int,
        num_classes: int,
        target_key: str,
        probe_train_loader,
        probe_val_loader, # =None,
        class_weights: Tensor | None = None,
        lr: float = 1e-2,
        n_epochs: int = 10,
        min_train_loss: float = 0.2,
    ) -> None:
        super().__init__()

        if class_weights is not None:
            assert class_weights.size(0) == self.num_classes, "number of class weights must equal `num_classes`"

        self.encoder_output_dim = encoder_output_dim
        self.num_classes = num_classes
        self.target_key = target_key
        self.class_weights = class_weights
        self.lr = lr
        self.n_epochs = n_epochs
        self.min_train_loss = min_train_loss

        self.probe_train_loader = probe_train_loader
        self.probe_val_loader = probe_val_loader

        self.optimizer: torch.optim.Optimizer

    def _init_model_opt(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:        
        # finetuner layer becomes a part of the main (embedding) model
        self.finetuner = nn.Linear(self.encoder_output_dim, self.num_classes).to(pl_module.device)
        # init optimizer inside the callback
        self.optimizer = torch.optim.Adam(self.finetuner.parameters(), lr=self.lr)

    def _encode_run(self, pl_module, run_mz, run_I):
        run_mz, run_I = run_mz.to(pl_module.device), run_I.to(pl_module.device)
        with torch.no_grad():
            peak_embs = pl_module.forward(run_mz, run_I)
            spec_embs = peak_embs.mean(dim=1) # average peak embeddings
            spec_embs = spec_embs.unsqueeze(dim=0) # (1, T, d)
            run_emb = spec_embs.mean(dim=1) # (1, d)
        return run_emb

    def _step(self, pl_module, batch, train=True):
        # extract components from batch
        runs_mz = batch["mz_array"]
        runs_I = batch["intensity_array"]
        targets = batch[self.target_key].to(pl_module.device)

        runs_emb = [
            self._encode_run(pl_module, runs_mz[i], runs_I[i])
            for i in range(len(runs_mz))
        ]
        embs = torch.cat(runs_emb, dim=0) 

        # apply fine-tuner model
        preds = self.finetuner(embs)

        # compute loss
        loss = F.cross_entropy(preds, targets, weight=self.class_weights)

        if train:
            loss.backward()
            self.optimizer.step()
            self.optimizer.zero_grad()
        
        acc = accuracy(preds, targets, task="multiclass", num_classes=self.num_classes)

        # Display prediction sample
        if not train:
            n = 30
            sample_df = np.hstack((
                targets[:n].cpu().numpy()[:, None],
                F.softmax(preds, dim=1)[:n].detach().cpu().numpy(),
            ))
            sample_df = pd.DataFrame(
                sample_df, columns=[
                    "targets", 
                    "prob_0", 
                    "prob_1", 
                    # "prob_2", 
                    # "prob_3", 
                ]
            )
            display(sample_df)
        return loss.detach(), acc.detach()

    def on_train_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Init the probe head and train for probe_n_epochs passes
        over probe_train_loader, after SSL epoch ends.
        """
        self._init_model_opt(trainer, pl_module)

        was_training = pl_module.training
        pl_module.eval()
        self.finetuner.train()

        # train model for self.n_epochs on self.probe_train_loader
        probe_epoch = 0
        avg_loss, avg_acc = 0.0, 0.0
        
        while (avg_loss > self.min_train_loss) and (
            self.n_epochs is None or probe_epoch < self.n_epochs
        ):            
            total_loss, total_acc, n_batches = 0.0, 0.0, 0
            for batch in self.probe_train_loader:
                loss, acc = self._step(pl_module, batch, train=True)
                total_loss += float(loss)
                total_acc += float(acc)
                n_batches += 1
    
            avg_loss = total_loss / n_batches
            avg_acc = total_acc / n_batches
            print(f"Probe epoch {probe_epoch} loss:", avg_loss)
            print(f"Probe epoch {probe_epoch} acc:", avg_acc)
            probe_epoch += 1

        # Log final train metrics
        pl_module.log("online_train_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_train_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

    def on_validation_epoch_end(self, trainer: L.Trainer, pl_module: L.LightningModule) -> None:
        """
        Train the probe head for one pass over probe_train_loader, after SSL epoch ends.
        """
        # workaround for num_sanity_val_steps>0
        if not hasattr(self, "finetuner"):
            self._init_model_opt(trainer, pl_module)
        
        was_training = pl_module.training
        pl_module.eval()
        self.finetuner.eval()

        total_loss, total_acc, n_batches = 0.0, 0.0, 0

        with torch.no_grad():
            for batch in self.probe_val_loader:
                loss, acc = self._step(pl_module, batch, train=False)
                total_loss += float(loss)
                total_acc += float(acc)
                n_batches += 1

        avg_loss = total_loss / n_batches
        avg_acc = total_acc / n_batches

        # Log epoch metrics
        pl_module.log("online_val_loss", avg_loss, on_step=False, on_epoch=True, prog_bar=False)
        pl_module.log("online_val_acc", avg_acc, on_step=False, on_epoch=True, prog_bar=False)

        if was_training:
            pl_module.train()

In [50]:
config

ExperimentConfig(name='ms1-mz-200_peaks-sqrt-clf_run_retrain', data=DataConfig(train_dir='train_mzml', val_dir='val_mzml', batch_size=2000, max_num_peaks=200), model=ModelConfig(d_model=256, nhead=8, dim_feedforward=512, n_layers=6, dropout=0.1, n_bins=1200, bin_mz_min=300, bin_mz_max=1500, masked_peaks_fraction=0.3), optimizer=OptimizerConfig(lr=0.0005, warmup_iters=1000, cosine_schedule_period_iters=64000), training=TrainingConfig(checkpoint_path='./train_checkpoints', max_epochs=1000, gradient_clip_val=5, accelerator='gpu', devices=1))

In [51]:
model = MS1Encoder(
    d_model=config.model.d_model,
    nhead=config.model.nhead,
    dim_feedforward=config.model.dim_feedforward,
    n_layers=config.model.n_layers,
    dropout=config.model.dropout,
    n_bins=config.model.n_bins,
    bin_mz_min=config.model.bin_mz_min,
    bin_mz_max=config.model.bin_mz_max,
    masked_peaks_fraction=config.model.masked_peaks_fraction,
    lr=config.optimizer.lr,
    warmup_iters=config.optimizer.warmup_iters,
    cosine_schedule_period_iters=config.optimizer.cosine_schedule_period_iters,
)

In [52]:
root_dir = os.path.join(config.training.checkpoint_path, "foundation_model")
os.makedirs(root_dir, exist_ok=True)

logger = L.loggers.TensorBoardLogger(
    os.path.join(root_dir, "lightning_logs"),
    name=config.name,
)

In [53]:
# TODO: number of classes needs to be parametrized (based on number of classes in val subset)
# and validation classes need to be "renamed" (mapped) to always be 0 < C < n_classes
meta_df.filter(meta_df["id"].is_in(val_ids))["genus_class"].value_counts()

genus_class,count
i64,u32
0,15
1,15


In [58]:
retrain_finetuner = FineTuner(
    encoder_output_dim=config.model.d_model, 
    num_classes=2,#len(genus_class),
    target_key="label",
    probe_train_loader=probe_train_loader,
    probe_val_loader=probe_val_loader,
    class_weights=None,
    lr=1e-2,
    n_epochs=100,
    min_train_loss=0.3,
)
# checkpoint_callback = L.ModelCheckpoint(every_n_epochs=100, save_top_k=-1, save_last=True)

trainer = L.Trainer(
    logger=logger,
    default_root_dir=root_dir,
    callbacks=[retrain_finetuner], #checkpoint_callback],
    accelerator="gpu", #config.training.accelerator,
    devices=config.training.devices,
    max_epochs=500, #config.training.max_epochs,
    gradient_clip_val=config.training.gradient_clip_val,
    num_sanity_val_steps=2,
    log_every_n_steps=10,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
# Train the model
trainer.fit(
    model, 
    train_loader, 
    val_dataloaders=[val_loader],
    # ckpt_path=ckpt_path, # 
)


Validation DataLoader 0: 100%|██████████| 44/44 [00:40<00:00,  1.08it/s]

In [ ]:
# 1e-1 -- can it be too large? Training diverges after the first grad step


# 1e-2 -- model converges for early embeddings,
# but for later embeddings seems like it doesn't have enough time to converge?
# вообще возможно, что и с 1e-2 все немного расходится в начале, но это не точно


# so does it mean, that we have to train not for a fixed number of epochs
# but till a fixed TRAIN loss value? 